# LaunchReady: memory validation on Kaggle T4 x2

**Does `launchready fit` predict real GPU memory?** This notebook checks the calculator's
projections against measurements on two NVIDIA Tesla T4s (16 GB each, Turing) for three
architectures from BUILD_PLAN.md Phase 3:

| Model | Kind | Why it's here |
|---|---|---|
| `Qwen/Qwen3-4B` | dense | baseline GQA transformer |
| `allenai/OLMoE-1B-7B-0924-Instruct` | MoE | memory scales with *total* parameters |
| `ibm-granite/granite-4.0-h-tiny` | hybrid Mamba | fixed Mamba state instead of a growing KV cache |

For each model:

1. **Projected:** `launchready fit --json` (config.json and safetensors headers only, no weights).
2. **Measured, weights:** load in FP16 with 🤗 Transformers and read
   `torch.cuda.max_memory_allocated()` (summed over GPUs).
3. **Measured, KV cache:** start `vllm serve --dtype half` (T4 has no BF16) and parse its
   startup lines for KV-cache capacity and maximum concurrency, then shut it down.

Models over ~13 GB in FP16 run across both GPUs (`--tensor-parallel-size 2`).
Every row of `results/memory_validation.csv` is labeled **measured** or **projected** and carries
the GPU, driver, CUDA, engine versions, model revision and date.

**To run:** Kaggle → Settings → Accelerator **GPU T4 x2**, Internet **on** → *Run all*
(~45–75 min). An `HF_TOKEN` Kaggle secret is optional (none of these models is gated).
GPU work runs in child processes, so a model that fails on T4 is logged and skipped instead of
stopping the notebook.


## 1. Install pinned versions and record the environment

In [ ]:
import json
import platform
import re
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

# Pinned versions: change them here and nowhere else. Every result row records them.
LAUNCHREADY_COMMIT = "689821dd3c9275702563c4c13f5b976f6bdb4c0b"
PINS = [
    "launchready @ git+https://github.com/Tanmaypathak18/launchready@" + LAUNCHREADY_COMMIT,
    "torch==2.13.0",  # vLLM 0.32.0 pins this exact torch
    "vllm==0.32.0",
    "transformers==5.19.0",  # inside vLLM 0.32.0's >=5.16.1,<5.20
    "accelerate==1.15.0",
]

RESULTS_DIR = Path("results")
LOG_DIR = RESULTS_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)


def run(cmd: list[str], timeout: int | None = None) -> str:
    return subprocess.run(cmd, capture_output=True, text=True, check=True, timeout=timeout).stdout


try:
    smi = run(["nvidia-smi"])
except (FileNotFoundError, subprocess.CalledProcessError) as e:
    raise RuntimeError(
        "No NVIDIA GPU visible. In the notebook's right-hand panel open Settings → Accelerator "
        "and pick 'GPU T4 x2', then Run all again."
    ) from e
print(smi)

print("Installing pinned packages (5–10 min the first time)...")
pip = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--progress-bar", "off", *PINS],
    capture_output=True,
    text=True,
)
if pip.returncode:
    print(pip.stdout[-4000:], pip.stderr[-4000:])
    raise RuntimeError("pip install failed (see above). Is Internet on in Settings?")

# Probe versions in a child process so this kernel never initializes CUDA: the GPUs stay
# fully free for the measurement subprocesses below.
PROBE = """
import importlib.metadata as md, json, sys
import torch
out = {"python": sys.version.split()[0], "torch": torch.__version__,
       "torch_cuda": torch.version.cuda, "cuda_available": torch.cuda.is_available(),
       "gpu_count": torch.cuda.device_count()}
for pkg in ("vllm", "transformers", "accelerate", "launchready"):
    out[pkg] = md.version(pkg)
direct = json.loads(md.distribution("launchready").read_text("direct_url.json") or "{}")
out["launchready_commit"] = direct.get("vcs_info", {}).get("commit_id")
print(json.dumps(out))
"""
probe = json.loads(run([sys.executable, "-c", PROBE], timeout=600).strip().splitlines()[-1])
gpus = [
    [field.strip() for field in line.split(",")]
    for line in run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader,nounits",
        ]
    )
    .strip()
    .splitlines()
]
ENV = {
    **probe,
    "gpu": gpus[0][0],
    "gpu_count_smi": len(gpus),
    "vram_mib": min(int(g[1]) for g in gpus),
    "driver": gpus[0][2],
    "driver_cuda": (re.search(r"CUDA Version:\s*([\d.]+)", smi) or [None, None])[1],
    "platform": platform.platform(),
    "date_utc": datetime.now(UTC).isoformat(timespec="seconds"),
}
(RESULTS_DIR / "environment.json").write_text(json.dumps(ENV, indent=2) + "\n")
print(json.dumps(ENV, indent=2))
print(run(["df", "-h", "/kaggle/working" if Path("/kaggle/working").exists() else "."]))

if not ENV["cuda_available"] or ENV["gpu_count"] < 2:
    raise RuntimeError(
        f"PyTorch sees {ENV['gpu_count']} GPU(s), cuda_available={ENV['cuda_available']}. "
        "Pick 'GPU T4 x2' in Settings → Accelerator. If both GPUs show in nvidia-smi above, "
        f"driver {ENV['driver']} (CUDA {ENV['driver_cuda']}) may be too old for torch "
        f"{ENV['torch']} (CUDA {ENV['torch_cuda']}): log it in docs/friction_log.md, then "
        "change the pins."
    )

## 2. Settings

In [ ]:
import os

MODELS = [
    ("Qwen/Qwen3-4B", "dense"),
    ("allenai/OLMoE-1B-7B-0924-Instruct", "MoE"),
    ("ibm-granite/granite-4.0-h-tiny", "hybrid Mamba"),
]
MAX_MODEL_LEN = 4096  # same context for every model so concurrency numbers compare
TP_THRESHOLD_BYTES = 13e9  # FP16 weights above this run across both GPUs
VLLM_PORT = 8000
VLLM_TIMEOUT_S = 1200  # startup includes download check, compile and CUDA-graph capture
HF_TIMEOUT_S = 1200
CSV_PATH = RESULTS_DIR / "memory_validation.csv"
PLOT_PATH = RESULTS_DIR / "memory_validation.png"

# Optional: a Kaggle secret named HF_TOKEN (Add-ons → Secrets). Never printed.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception:
    print("No HF_TOKEN secret attached; fine for these public models")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

## 3. Helpers: projection, Transformers, vLLM, CSV rows

In [ ]:
import os
import signal
import tempfile
import time
import urllib.request

from launchready.memory import GIB, GPU_MEMORY_UTILIZATION, RUNTIME_RESERVE_BYTES, fit

VRAM_BYTES = ENV["vram_mib"] * 2**20  # 6.3: measured GPUs use nvidia-smi VRAM, not the catalog
SCRIPT_DIR = Path(tempfile.gettempdir()) / "launchready_nb"
SCRIPT_DIR.mkdir(exist_ok=True)


class StageError(RuntimeError):
    """A stage failed; the message carries the reason and the tail of its log."""


def slug(repo_id: str) -> str:
    return repo_id.replace("/", "__")


def tail(text: str, lines: int = 20) -> str:
    return "\n".join(text.strip().splitlines()[-lines:])


def reason(e: Exception) -> str:
    return f"{type(e).__name__}: {e}"[:2000]


# --- Projected: launchready fit -------------------------------------------------------------


def project(repo_id: str) -> dict:
    """launchready's projection, re-fit to the measured T4 VRAM and the chosen TP size."""
    cli = "from launchready.cli import app; app()"
    args = ["fit", repo_id, "--json", "--gpu", "tesla-t4", "--context", str(MAX_MODEL_LEN)]
    out = subprocess.run(
        [sys.executable, "-c", cli, *args, "--users", "1"],
        capture_output=True,
        text=True,
        timeout=600,
    )
    if out.returncode:
        raise StageError(f"launchready fit exited {out.returncode}: {tail(out.stderr)}")
    (LOG_DIR / f"{slug(repo_id)}__fit.json").write_text(out.stdout)
    proj = json.loads(out.stdout)
    weights = proj["weights_bytes"].get("bf16")
    if weights is None:
        raise StageError("no safetensors parameter count, so no weight projection")
    tp = 2 if weights > TP_THRESHOLD_BYTES else 1
    seq = proj["sequence"]
    # 6.3 across tp GPUs: KV budget = tp × VRAM × 0.90 − weights − tp × reserve.
    total = fit(
        tp * VRAM_BYTES, weights, seq["total_bytes"], reserve_bytes=tp * RUNTIME_RESERVE_BYTES
    )
    budget = max(total.kv_budget_bytes, 0)
    return {
        "revision": proj["revision"],
        "tp": tp,
        "weights_bytes": weights,
        "kv_budget_bytes_per_gpu": total.kv_budget_bytes / tp,
        "kv_cache_tokens": budget / seq["kv_bytes_per_token"]
        if seq["kv_bytes_per_token"]
        else None,
        "max_concurrency": budget / seq["total_bytes"],
        "fit": proj,
    }


# --- Download once, shared by Transformers and vLLM; deleted after each model --------------


def download(repo_id: str, revision: str) -> None:
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id,
        revision=revision,
        allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja"],
    )


def delete_download(repo_id: str) -> None:
    from huggingface_hub import scan_cache_dir

    cache = scan_cache_dir()
    hashes = [
        r.commit_hash for repo in cache.repos if repo.repo_id == repo_id for r in repo.revisions
    ]
    if hashes:
        cache.delete_revisions(*hashes).execute()


# --- Measured: Transformers + torch.cuda.max_memory_allocated (child process) ---------------

MEASURE_HF = SCRIPT_DIR / "measure_hf.py"
MEASURE_HF.write_text(
    """
import json, sys, time
import torch
from transformers import AutoModelForCausalLM

repo_id, revision, tp = sys.argv[1], sys.argv[2], int(sys.argv[3])
devices = list(range(tp))
for d in devices:
    torch.cuda.reset_peak_memory_stats(d)
start = time.time()
model = AutoModelForCausalLM.from_pretrained(
    repo_id,
    revision=revision,
    dtype=torch.float16,
    device_map={"": 0} if tp == 1 else "auto",
    max_memory=None if tp == 1 else {d: "14GiB" for d in devices},
)
for d in devices:
    torch.cuda.synchronize(d)
print("RESULT " + json.dumps({
    "max_memory_allocated": [torch.cuda.max_memory_allocated(d) for d in devices],
    "memory_allocated": [torch.cuda.memory_allocated(d) for d in devices],
    "parameter_bytes": sum(p.numel() * p.element_size() for p in model.parameters()),
    "load_seconds": round(time.time() - start, 1),
}))
"""
)


def measure_hf(repo_id: str, revision: str, tp: int) -> dict:
    log_path = LOG_DIR / f"{slug(repo_id)}__transformers.log"
    try:
        out = subprocess.run(
            [sys.executable, str(MEASURE_HF), repo_id, revision, str(tp)],
            capture_output=True,
            text=True,
            timeout=HF_TIMEOUT_S,
        )
    except subprocess.TimeoutExpired as e:
        raise StageError(f"Transformers load timed out after {HF_TIMEOUT_S}s") from e
    log_path.write_text(out.stdout + "\n--- stderr ---\n" + out.stderr)
    results = [line for line in out.stdout.splitlines() if line.startswith("RESULT ")]
    if out.returncode or not results:
        raise StageError(f"Transformers exited {out.returncode}: {tail(out.stderr)}")
    hf = json.loads(results[-1].removeprefix("RESULT "))
    hf["peak_bytes"] = sum(hf["max_memory_allocated"])
    return hf


# --- Measured: vLLM startup log lines (6.8) --------------------------------------------------

VLLM_PATTERNS = {
    "kv_cache_tokens": r"GPU KV cache size:\s*([\d,]+)\s*tokens",
    "max_concurrency": r"Maximum concurrency for\s*[\d,]+\s*tokens per request:\s*([\d.]+)x",
    "model_loading_gib": r"Model loading took\s*([\d.]+)\s*GiB",
    "available_kv_gib": r"Available KV cache memory:\s*([\d.]+)\s*GiB",
    "gpu_blocks": r"# GPU blocks:\s*([\d,]+)",  # older V0 engine
}


def parse_vllm_log(text: str, tp: int = 1) -> dict:
    """Pull 6.8's startup numbers out of a vLLM log. Missing lines come back as None."""
    found = {
        key: [float(m.replace(",", "")) for m in re.findall(pattern, text)]
        for key, pattern in VLLM_PATTERNS.items()
    }
    loading = found["model_loading_gib"]
    # Each TP worker logs its own shard; if only rank 0 logs, scale it by tp.
    weights_gib = sum(loading) if len(loading) == tp else (loading[0] * tp if loading else None)
    tokens = min(found["kv_cache_tokens"]) if found["kv_cache_tokens"] else None
    if tokens is None and found["gpu_blocks"]:
        tokens = min(found["gpu_blocks"]) * 16  # V0 default block size
    return {
        "kv_cache_tokens": tokens,
        "max_concurrency": min(found["max_concurrency"]) if found["max_concurrency"] else None,
        "weights_bytes": weights_gib * GIB if weights_gib is not None else None,
        "kv_budget_bytes_per_gpu": (
            min(found["available_kv_gib"]) * GIB if found["available_kv_gib"] else None
        ),
    }


def gpu_used_mib() -> list[int]:
    out = run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"])
    return [int(x) for x in out.split()]


def wait_gpus_free(timeout: int = 180, limit_mib: int = 1024) -> None:
    deadline = time.time() + timeout
    while time.time() < deadline:
        if max(gpu_used_mib()) < limit_mib:
            return
        time.sleep(3)
    raise StageError(f"GPU memory not released after {timeout}s: {gpu_used_mib()} MiB used")


def vllm_healthy() -> bool:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{VLLM_PORT}/health", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False


def vllm_generates(repo_id: str) -> bool:
    body = json.dumps({"model": repo_id, "prompt": "The capital of France is", "max_tokens": 8})
    request = urllib.request.Request(
        f"http://127.0.0.1:{VLLM_PORT}/v1/completions",
        data=body.encode(),
        headers={"Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(request, timeout=120) as r:
            return bool(json.load(r)["choices"][0]["text"])
    except Exception:
        return False


def stop(proc: subprocess.Popen) -> None:
    """SIGTERM the whole process group (API server + TP workers), then SIGKILL stragglers."""
    for sig, wait in ((signal.SIGTERM, 60), (signal.SIGKILL, 30)):
        try:
            os.killpg(proc.pid, sig)
        except ProcessLookupError:
            return
        try:
            proc.wait(timeout=wait)
            return
        except subprocess.TimeoutExpired:
            continue


def run_vllm(repo_id: str, revision: str, tp: int) -> dict:
    log_path = LOG_DIR / f"{slug(repo_id)}__vllm.log"
    cmd = [
        sys.executable, "-c", "from vllm.entrypoints.cli.main import main; main()",
        "serve", repo_id,
        "--revision", revision,
        "--tokenizer-revision", revision,
        "--dtype", "half",
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--tensor-parallel-size", str(tp),
        "--port", str(VLLM_PORT),
    ]  # fmt: skip
    print("  $ vllm " + " ".join(cmd[3:]))
    with log_path.open("w") as log:
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    try:
        deadline, healthy = time.time() + VLLM_TIMEOUT_S, False
        while time.time() < deadline and proc.poll() is None:
            if vllm_healthy():
                healthy = True
                break
            time.sleep(5)
        generates = healthy and vllm_generates(repo_id)
    finally:
        stop(proc)
    text = log_path.read_text(errors="replace")
    result = parse_vllm_log(text, tp)
    result["generates"] = generates
    if not healthy:
        why = (
            f"exited {proc.returncode}" if proc.returncode else f"not healthy in {VLLM_TIMEOUT_S}s"
        )
        result["error"] = f"vLLM {why}: {tail(text)}"
    elif not generates:
        result["error"] = "vLLM started but the test completion failed"
    return result


# --- CSV rows --------------------------------------------------------------------------------

ENV_COLUMNS = ["gpu", "gpu_count", "driver", "driver_cuda", "torch_cuda", "torch", "transformers",
               "vllm", "launchready_commit", "date_utc"]  # fmt: skip
SOURCES = {
    ("weights_bytes", "projected"): "launchready fit: safetensors params × 16 bits ÷ 8",
    ("weights_bytes", "measured"): "Transformers FP16 load: sum of torch.cuda.max_memory_allocated",
    ("vllm_weights_bytes", "measured"): "vLLM log: 'Model loading took X GiB' (all TP ranks)",
    ("kv_budget_bytes_per_gpu", "projected"): "VRAM × 0.90 − weights ÷ tp − 2 GiB reserve",
    ("kv_budget_bytes_per_gpu", "measured"): "vLLM log: 'Available KV cache memory'",
    ("kv_cache_tokens", "projected"): "tp × per-GPU KV budget ÷ KV bytes per token",
    ("kv_cache_tokens", "measured"): "vLLM log: 'GPU KV cache size: N tokens'",
    ("max_concurrency", "projected"): (
        "KV budget ÷ per-sequence memory (KV + Mamba state) at max_model_len"
    ),
    ("max_concurrency", "measured"): "vLLM log: 'Maximum concurrency for N tokens per request'",
    ("vllm_generates", "measured"): "8-token test completion against the running server (1 = ok)",
}
UNITS = {"max_concurrency": "sequences", "kv_cache_tokens": "tokens", "vllm_generates": "bool"}


def make_rows(repo_id: str, kind: str, proj: dict | None, hf: dict | None, vl: dict | None,
              errors: dict) -> list[dict]:  # fmt: skip
    """One row per metric × label; a missing value keeps its row with status and reason."""
    vl = vl or {}
    values = {
        ("weights_bytes", "projected"): (proj, "weights_bytes", "projection"),
        ("weights_bytes", "measured"): (hf, "peak_bytes", "transformers"),
        ("vllm_weights_bytes", "measured"): (vl, "weights_bytes", "vllm"),
        ("kv_budget_bytes_per_gpu", "projected"): (proj, "kv_budget_bytes_per_gpu", "projection"),
        ("kv_budget_bytes_per_gpu", "measured"): (vl, "kv_budget_bytes_per_gpu", "vllm"),
        ("kv_cache_tokens", "projected"): (proj, "kv_cache_tokens", "projection"),
        ("kv_cache_tokens", "measured"): (vl, "kv_cache_tokens", "vllm"),
        ("max_concurrency", "projected"): (proj, "max_concurrency", "projection"),
        ("max_concurrency", "measured"): (vl, "max_concurrency", "vllm"),
        ("vllm_generates", "measured"): (vl, "generates", "vllm"),
    }
    rows = []
    for (metric, label), (source, key, stage) in values.items():
        value = source.get(key) if source else None
        if isinstance(value, bool):
            value = int(value)
        if value is not None:
            status, why = "ok", ""
        elif stage in errors:
            status, why = "failed", errors[stage]
        elif source:
            status, why = "missing", "line not in vLLM log (or not applicable to this model)"
        else:
            status, why = "skipped", next(iter(errors.values()), "earlier stage failed")
        rows.append({
            "repo_id": repo_id,
            "revision": proj["revision"] if proj else "",
            "model_kind": kind,
            "metric": metric,
            "label": label,
            "value": value,
            "unit": UNITS.get(metric, "bytes"),
            "source": SOURCES[(metric, label)],
            "tp": proj["tp"] if proj else "",
            "max_model_len": MAX_MODEL_LEN,
            "status": status,
            "reason": why,
            "vram_bytes": VRAM_BYTES,
            **{c: ENV.get(c) for c in ENV_COLUMNS},
        })  # fmt: skip
    return rows

Quick self-test of the vLLM log parser on sample startup lines, so a regex mismatch shows up
before any GPU time is spent.


In [ ]:
SAMPLE_LOG = """
(EngineCore_DP0 pid=812) INFO [gpu_model_runner.py:2602]
Model loading took 7.5508 GiB memory and 38.1 seconds
(EngineCore_DP0 pid=812) INFO [gpu_worker.py:298] Available KV cache memory: 4.12 GiB
(EngineCore_DP0 pid=812) INFO [kv_cache_utils.py:1087] GPU KV cache size: 29,984 tokens
(EngineCore_DP0 pid=812) INFO [kv_cache_utils.py:1091]
Maximum concurrency for 4,096 tokens per request: 7.32x
"""
parsed = parse_vllm_log(SAMPLE_LOG)
assert parsed["kv_cache_tokens"] == 29_984, parsed
assert parsed["max_concurrency"] == 7.32, parsed
assert abs(parsed["weights_bytes"] - 7.5508 * GIB) < 1, parsed
assert abs(parsed["kv_budget_bytes_per_gpu"] - 4.12 * GIB) < 1, parsed
rank0_only = SAMPLE_LOG.replace("took 7.5508", "took 3.8")
tp2 = parse_vllm_log(rank0_only, tp=2)
assert abs(tp2["weights_bytes"] - 7.6 * GIB) < 1, tp2  # rank 0 only → scaled by tp
assert parse_vllm_log("nothing useful")["kv_cache_tokens"] is None
print("parser self-test passed")

## Run every model

Each stage is isolated: if one fails (for example, a kernel that needs a newer GPU than Turing),
its reason is recorded and the notebook moves on. The CSV is rewritten after every model, so
partial results survive even if the session is cut off.


In [ ]:
import pandas as pd

rows = []
for repo_id, kind in MODELS:
    print(f"\n=== {repo_id} ({kind}) ===")
    errors, proj, hf, vl = {}, None, None, None
    try:
        wait_gpus_free()
        proj = project(repo_id)
        print(f"  projected FP16 weights {proj['weights_bytes'] / GIB:.2f} GiB → tp={proj['tp']}"
              f" · revision {proj['revision'][:7]}")  # fmt: skip
    except Exception as e:
        errors["projection"] = reason(e)
    if proj:
        try:
            download(repo_id, proj["revision"])
        except Exception as e:
            errors["transformers"] = errors["vllm"] = f"download failed: {reason(e)}"
        else:
            try:
                hf = measure_hf(repo_id, proj["revision"], proj["tp"])
                print(
                    f"  Transformers peak {hf['peak_bytes'] / GIB:.2f} GiB in {hf['load_seconds']}s"
                )
            except Exception as e:
                errors["transformers"] = reason(e)
            try:
                wait_gpus_free()
                vl = run_vllm(repo_id, proj["revision"], proj["tp"])
                if "error" in vl:
                    errors["vllm"] = vl["error"]
                print(
                    f"  vLLM KV cache {vl['kv_cache_tokens']} tokens · max concurrency "
                    f"{vl['max_concurrency']} · generates={vl['generates']}"
                )
            except Exception as e:
                errors["vllm"] = reason(e)
        finally:
            try:
                delete_download(repo_id)  # Kaggle's disk can't hold all three models at once
            except Exception as e:
                print(f"  could not delete the download: {reason(e)}")
    for stage, why in errors.items():
        print(f"  SKIPPED {stage}: {why.splitlines()[0]}  (full reason: CSV, results/logs/)")
    rows += make_rows(repo_id, kind, proj, hf, vl, errors)
    pd.DataFrame(rows).to_csv(CSV_PATH, index=False)

results = pd.DataFrame(rows)
print(f"\nwrote {CSV_PATH} ({len(results)} rows)")

## Projected vs measured

Error = (projected − measured) ÷ measured. Phase 3 targets: weights within ~5%, KV capacity
within ~10–15%. Bytes are stored in the CSV; GiB is for display only.


In [ ]:
COMPARED = ["weights_bytes", "kv_budget_bytes_per_gpu", "kv_cache_tokens", "max_concurrency"]
ok = results[(results.status == "ok") & results.metric.isin(COMPARED)]
comparison = ok.pivot_table(
    index=["repo_id", "metric"], columns="label", values="value", aggfunc="first"
)
comparison = comparison.reindex(columns=["projected", "measured"])
comparison["error_%"] = 100 * (comparison.projected - comparison.measured) / comparison.measured
shown = comparison.copy()
in_bytes = shown.index.get_level_values("metric").str.contains("bytes")
shown.loc[in_bytes, ["projected", "measured"]] /= GIB
shown = shown.rename(index=lambda m: m.replace("bytes", "GiB"), level="metric")
display(shown.round(2))

problems = results[results.status != "ok"][["repo_id", "metric", "label", "status", "reason"]]
if len(problems):
    print("Not measured (reasons are in the CSV; logs in results/logs/):")
    display(problems.assign(reason=problems.reason.str.split("\n").str[0].str[:160]))

In [ ]:
import matplotlib.pyplot as plt

# Reference categorical palette, first three slots (validated for all pairs, light surface).
COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]
model_color = {repo: COLORS[i] for i, (repo, _) in enumerate(MODELS)}
# Per-model label offsets so the % labels don't collide when points sit close together.
label_offset = {repo: [(8, 6), (-38, 8), (8, -14)][i] for i, (repo, _) in enumerate(MODELS)}
PANELS = [
    ("weights_bytes", "Weights (GiB)", GIB, 5),
    ("kv_cache_tokens", "KV-cache capacity (tokens)", 1, 15),
    ("max_concurrency", f"Max concurrency at {MAX_MODEL_LEN:,} tokens", 1, 15),
]

if comparison.dropna(subset=["projected", "measured"]).empty:
    print("Nothing to plot: no metric has both a projected and a measured value (see above).")
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
    for ax, (metric, title, scale, band) in zip(axes, PANELS, strict=True):
        data = comparison.xs(metric, level="metric").dropna(subset=["projected", "measured"])
        ax.set_title(f"{title} · band ±{band}%", loc="left", fontsize=11)
        ax.set_xlabel("measured")
        ax.set_ylabel("projected")
        ax.grid(color="#e4e3df", linewidth=0.8)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        if data.empty:
            ax.text(0.5, 0.5, "not measured", ha="center", va="center", transform=ax.transAxes,
                    color="#52514e")  # fmt: skip
            continue
        x, y = data.measured / scale, data.projected / scale
        hi = max(x.max(), y.max()) * 1.15
        line = [0, hi]
        ax.plot(line, line, color="#52514e", linewidth=1, label="projected = measured")
        ax.fill_between(line, [0, hi * (1 - band / 100)], [0, hi * (1 + band / 100)],
                        color="#52514e", alpha=0.08, linewidth=0)  # fmt: skip
        for repo, xi, yi, err in zip(data.index, x, y, data["error_%"], strict=True):
            ax.scatter(xi, yi, s=70, color=model_color[repo], edgecolor="white", linewidth=2,
                       zorder=3, label=repo)  # fmt: skip
            ax.annotate(
                f"{err:+.0f}%",
                (xi, yi),
                xytext=label_offset[repo],
                textcoords="offset points",
                fontsize=9,
                color="#0b0b0b",
            )
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
        ax.set_xlim(0, hi)
        ax.set_ylim(0, hi)
    handles = {h.get_label(): h for ax in axes for h in ax.get_legend_handles_labels()[0]}
    fig.legend(handles.values(), handles.keys(), loc="lower center", ncol=4, frameon=False)
    fig.suptitle(
        f"launchready fit: projected vs measured · {ENV['gpu_count']}× {ENV['gpu']} · "
        f"vLLM {ENV['vllm']} · driver {ENV['driver']} · {ENV['date_utc'][:10]}",
        x=0.01, ha="left", fontsize=12,
    )  # fmt: skip
    fig.tight_layout(rect=(0, 0.08, 1, 0.95))
    fig.savefig(PLOT_PATH, dpi=150)
    plt.show()
    print(f"wrote {PLOT_PATH}")

## Reading the misses

- **Weights:** Transformers' peak includes small load-time buffers; vLLM's "Model loading"
  line is a second measured reference. A miss well over 5% points at the parameter count.
- **KV budget / capacity:** the projection subtracts a flat 2 GiB runtime reserve per GPU
  (BUILD_PLAN 6.3). vLLM's real reserve (activations, CUDA graphs, NCCL buffers under TP=2) is
  the most likely miss; `kv_budget_bytes_per_gpu` measured vs projected calibrates it directly.
- **Hybrid Mamba (granite-4.0-h-tiny):** launchready assumes FP32 Mamba state, and vLLM packs
  attention KV and Mamba state into shared pages, so its token count isn't directly comparable to
  KV bytes per token. Compare **max concurrency**, which counts both.

## When something breaks

Follow CLAUDE.md: **write a `docs/friction_log.md` entry (template 7.3) before changing anything**,
using the log in `results/logs/` and the environment in `results/environment.json`. Only then
change the vLLM pin or swap the model, and say so in the entry. Never swap silently.

Finally, download `results/` (CSV, chart, environment.json, logs) and commit it next to this
notebook.
